# Qdrant Ingestion Pipeline — BGE-M3 (Dense + Sparse)

Reads pre-chunked JSON files (`*_chunks.json`) and stores them in a local Qdrant vector database with BGE-M3 dense + sparse embeddings.

**Expected layout:**
```
project/
├── ingest.ipynb
├── chunks/
│   ├── AAL_2010_chunks.json
│   ├── DAL_2011_chunks.json
│   └── ...
└── qdrant_db/          ← created automatically
```

**Requirements:** `pip install qdrant-client FlagEmbedding tqdm`

## Configuration

In [ ]:
import json
import sys
import uuid
from pathlib import Path
from typing import Any

from tqdm.notebook import tqdm

# ── Configuration ─────────────────────────────────────────
CHUNKS_DIR = "./chunks"                  # Directory with *_chunks.json files
QDRANT_PATH = "./qdrant_db"             # Local Qdrant storage path
COLLECTION_NAME = "annual_reports"
BGE_M3_MODEL = "BAAI/bge-m3"
BATCH_SIZE_EMBED = 64                   # Adjust down if OOM
BATCH_SIZE_UPSERT = 100
RECREATE_COLLECTION = False             # Set True to wipe and recreate

UUID_NAMESPACE = uuid.NAMESPACE_URL
REQUIRED_FIELDS = {"chunk_id", "pdf_name", "type", "page", "text"}

## Step 1 — Load and validate chunks

In [ ]:
def load_chunks_from_file(filepath: Path) -> list[dict[str, Any]]:
    """Load and parse a single JSON chunk file."""
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"{filepath.name}: expected JSON array, got {type(data).__name__}")
    return data


def validate_chunk(chunk: dict[str, Any]) -> list[str]:
    """Validate a single chunk. Returns list of error messages (empty = valid)."""
    errors = []
    missing = REQUIRED_FIELDS - set(chunk.keys())
    if missing:
        errors.append(f"Missing fields: {missing}")
    if "type" in chunk and chunk["type"] not in ("text", "table"):
        errors.append(f"Invalid type: {chunk['type']!r}")
    if "page" in chunk and not isinstance(chunk["page"], int):
        errors.append(f"page should be int, got {type(chunk['page']).__name__}")
    return errors


def extract_ticker_year(pdf_name: str) -> tuple[str, int]:
    """Extract ticker and year from pdf_name like 'AAL_2010'."""
    parts = pdf_name.split("_")
    if len(parts) < 2:
        raise ValueError(f"Cannot parse pdf_name: {pdf_name!r}")
    return parts[0], int(parts[1])


def load_all_chunks(chunks_dir: str) -> list[dict[str, Any]]:
    """Load all *_chunks.json files from the directory."""
    chunks_path = Path(chunks_dir)
    if not chunks_path.exists():
        print(f"ERROR: Chunks directory not found: {chunks_dir}")
        return []

    json_files = sorted(chunks_path.glob("*_chunks.json"))
    if not json_files:
        print(f"ERROR: No *_chunks.json files found in {chunks_dir}")
        return []

    print(f"Found {len(json_files)} chunk file(s) in {chunks_dir}")

    all_chunks = []
    total_errors = 0

    for filepath in json_files:
        try:
            file_chunks = load_chunks_from_file(filepath)
        except (json.JSONDecodeError, ValueError) as e:
            print(f"  SKIP {filepath.name}: {e}")
            continue

        valid_count = 0
        for i, chunk in enumerate(file_chunks):
            errors = validate_chunk(chunk)
            if errors:
                print(f"  WARN {filepath.name}[{i}] chunk_id={chunk.get('chunk_id', '?')}: {errors}")
                total_errors += 1
                continue

            try:
                ticker, year = extract_ticker_year(chunk["pdf_name"])
                chunk["ticker"] = ticker
                chunk["year"] = year
            except ValueError as e:
                print(f"  WARN {filepath.name}[{i}]: {e}")
                total_errors += 1
                continue

            valid_count += 1
            all_chunks.append(chunk)

        print(f"  {filepath.name}: {valid_count}/{len(file_chunks)} chunks valid")

    print(f"\nTotal chunks loaded: {len(all_chunks)}  |  Errors: {total_errors}")

    if all_chunks:
        pdf_names = set(c["pdf_name"] for c in all_chunks)
        type_counts = {}
        for c in all_chunks:
            type_counts[c["type"]] = type_counts.get(c["type"], 0) + 1
        token_counts = [c.get("token_count", 0) for c in all_chunks]
        print(f"Documents: {len(pdf_names)}  |  Types: {type_counts}")
        print(f"Tokens: min={min(token_counts)}, max={max(token_counts)}, avg={sum(token_counts)/len(token_counts):.0f}")

    return all_chunks

In [ ]:
chunks = load_all_chunks(CHUNKS_DIR)

# Show a few samples
for c in chunks[:3]:
    print(f"  {c['chunk_id']} | type={c['type']} | page={c['page']} | tokens={c.get('token_count','?')}")
    print(f"    text: {c['text'][:100]}...")
    if c.get("html"):
        print(f"    html: {c['html'][:80]}...")
    print()

## Step 2 — Initialize Qdrant collection

In [ ]:
from qdrant_client import QdrantClient, models

def init_qdrant(qdrant_path: str, collection_name: str, recreate: bool = False) -> QdrantClient:
    """Initialize Qdrant client and create collection with schema."""
    client = QdrantClient(path=qdrant_path)
    print(f"Qdrant client initialized (path={qdrant_path})")

    existing = [c.name for c in client.get_collections().collections]

    if collection_name in existing:
        if recreate:
            print(f"Recreating collection '{collection_name}'...")
            client.delete_collection(collection_name)
        else:
            info = client.get_collection(collection_name)
            print(f"Collection '{collection_name}' exists — {info.points_count} points. Skipping creation.")
            return client

    client.create_collection(
        collection_name=collection_name,
        vectors_config={
            "dense": models.VectorParams(
                size=1024,
                distance=models.Distance.COSINE,
            ),
        },
        sparse_vectors_config={
            "sparse": models.SparseVectorParams(
                modifier=models.Modifier.IDF,
            ),
        },
    )
    print(f"Collection '{collection_name}' created (dense=1024d cosine, sparse=IDF)")

    # Payload indexes (effective in server mode, no-op in embedded mode)
    for field, schema_type in [
        ("pdf_name", models.PayloadSchemaType.KEYWORD),
        ("ticker",   models.PayloadSchemaType.KEYWORD),
        ("year",     models.PayloadSchemaType.INTEGER),
        ("type",     models.PayloadSchemaType.KEYWORD),
        ("section",  models.PayloadSchemaType.KEYWORD),
        ("page",     models.PayloadSchemaType.INTEGER),
    ]:
        client.create_payload_index(collection_name, field, schema_type)

    print("Payload indexes created (6 fields)")
    return client

In [ ]:
client = init_qdrant(QDRANT_PATH, COLLECTION_NAME, recreate=RECREATE_COLLECTION)

## Step 3 & 4 — Embed with BGE-M3 and upsert to Qdrant

In [ ]:
from FlagEmbedding import BGEM3FlagModel
from qdrant_client.models import PointStruct, SparseVector


def to_sparse_vector(lexical_weights: dict) -> SparseVector:
    """Convert BGE-M3 lexical_weights dict to Qdrant SparseVector."""
    if not lexical_weights:
        return SparseVector(indices=[], values=[])
    indices = list(lexical_weights.keys())
    values = list(lexical_weights.values())
    return SparseVector(
        indices=[int(i) for i in indices],
        values=[float(v) for v in values],
    )


def chunk_to_point_id(chunk_id: str) -> str:
    """Generate deterministic UUID5 from chunk_id."""
    return str(uuid.uuid5(UUID_NAMESPACE, chunk_id))


def build_payload(chunk: dict[str, Any]) -> dict[str, Any]:
    """Build Qdrant payload from chunk. Excludes markdown/serialized."""
    return {
        "chunk_id":    chunk["chunk_id"],
        "pdf_name":    chunk["pdf_name"],
        "ticker":      chunk["ticker"],
        "year":        chunk["year"],
        "type":        chunk["type"],
        "page":        chunk["page"],
        "section":     chunk.get("section"),
        "token_count": chunk.get("token_count"),
        "text":        chunk["text"],
        "html":        chunk.get("html"),
    }


def embed_and_upsert(
    client: QdrantClient,
    collection_name: str,
    chunks: list[dict[str, Any]],
    bge_model: BGEM3FlagModel,
    batch_size_embed: int = 64,
    batch_size_upsert: int = 100,
) -> None:
    """Generate embeddings and upsert all chunks into Qdrant."""
    print(f"Embedding and upserting {len(chunks)} chunks...")
    print(f"  Embed batch size:  {batch_size_embed}")
    print(f"  Upsert batch size: {batch_size_upsert}")

    points_buffer: list[PointStruct] = []
    upserted = 0

    for batch_start in tqdm(range(0, len(chunks), batch_size_embed), desc="Embedding"):
        batch = chunks[batch_start : batch_start + batch_size_embed]
        texts = [c["text"] for c in batch]

        output = bge_model.encode(
            texts,
            return_dense=True,
            return_sparse=True,
            return_colbert_vecs=False,
            batch_size=batch_size_embed,
        )

        for i, chunk in enumerate(batch):
            dense_vec = output["dense_vecs"][i].tolist()
            sparse_vec = to_sparse_vector(output["lexical_weights"][i])

            point = PointStruct(
                id=chunk_to_point_id(chunk["chunk_id"]),
                vector={
                    "dense": dense_vec,
                    "sparse": sparse_vec,
                },
                payload=build_payload(chunk),
            )
            points_buffer.append(point)

        # Flush upsert buffer
        while len(points_buffer) >= batch_size_upsert:
            batch_to_upsert = points_buffer[:batch_size_upsert]
            points_buffer = points_buffer[batch_size_upsert:]
            client.upsert(collection_name=collection_name, points=batch_to_upsert)
            upserted += len(batch_to_upsert)

    # Flush remaining
    if points_buffer:
        client.upsert(collection_name=collection_name, points=points_buffer)
        upserted += len(points_buffer)

    print(f"Upserted {upserted} points total.")

In [ ]:
print(f"Loading BGE-M3 model ({BGE_M3_MODEL})...")
bge_model = BGEM3FlagModel(BGE_M3_MODEL, use_fp16=True, device="cuda")
print("Model loaded on GPU.")

In [ ]:
embed_and_upsert(client, COLLECTION_NAME, chunks, bge_model, BATCH_SIZE_EMBED, BATCH_SIZE_UPSERT)

## Step 5 — Verification

In [ ]:
info = client.get_collection(COLLECTION_NAME)
print(f"Collection '{COLLECTION_NAME}'")
print(f"  Points:         {info.points_count}")
print(f"  Status:         {info.status}")
print(f"  Dense vectors:  {info.config.params.vectors}")
print(f"  Sparse vectors: {info.config.params.sparse_vectors}")

In [ ]:
# Test query — unfiltered
test_question = "What were the fuel costs?"
query_output = bge_model.encode([test_question], return_dense=True, return_sparse=True, return_colbert_vecs=False)
query_dense = query_output["dense_vecs"][0].tolist()

results = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_dense,
    using="dense",
    limit=5,
)

print(f"Query: '{test_question}'\n")
for r in results.points:
    print(f"  score={r.score:.4f}  {r.payload['chunk_id']}  type={r.payload['type']}  page={r.payload['page']}")
    print(f"    {r.payload['text'][:120]}...")
    print()

In [ ]:
# Test query — filtered by document
results_filtered = client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_dense,
    using="dense",
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="pdf_name",
                match=models.MatchValue(value="AAL_2010"),
            )
        ]
    ),
    limit=5,
)

print("Filtered query (pdf_name=AAL_2010):\n")
for r in results_filtered.points:
    print(f"  score={r.score:.4f}  {r.payload['chunk_id']}")

In [ ]:
# Cleanup — close Qdrant client when done
# client.close()